**Метод:** MapReduce  
**Библиотека:** `mrjob`  
**Входные данные:** текстовый файл  
**Основной расчет:** количество строк длиной от 30 до 40 символов

Дополнительно рассчитываются общее число строк, средняя длина строки
и распределение строк по диапазонам длины.

## 0. Установка библиотек

In [ ]:
!pip -q install mrjob

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

## 1. Загрузка текстового файла

In [ ]:
try:
    from google.colab import files

    uploaded = files.upload()

    if uploaded:
        file_name = next(iter(uploaded.keys()))
    else:
        file_name = None

except Exception:
    file_name = None


if file_name is None:
    test_content = """Короткая строка.
Это строка содержит примерно тридцать пять знаков.
Еще одна строка подходящей длины текста.
Совсем коротко.
Строка с количеством символов около сорока.
Это значительно более длинная строка, которая выходит далеко за пределы выбранного диапазона.
"""

    file_name = "test_file.txt"

    Path(file_name).write_text(
        test_content,
        encoding="utf-8"
    )


print("Файл:", file_name)
print(
    "Размер:",
    os.path.getsize(file_name),
    "байт"
)

## 2. MapReduce-скрипт

In [ ]:
mr_script = r"""
from mrjob.job import MRJob


class CountLines30to40(MRJob):
    def mapper(self, _, line):
        clean_line = line.rstrip("\\n")
        line_length = len(clean_line)

        if 30 <= line_length <= 40:
            yield "lines_30_to_40", 1

    def reducer(self, key, values):
        yield key, sum(values)


if __name__ == "__main__":
    CountLines30to40.run()
"""

script_name = "count_lines_30_40.py"

Path(script_name).write_text(
    mr_script,
    encoding="utf-8"
)

print("Создан файл:", script_name)

## 3. Запуск MapReduce

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        script_name,
        file_name,
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
)

print("STDOUT:")
print(result.stdout)

if result.stderr.strip():
    print("STDERR:")
    print(result.stderr)

## 4. Контрольный подсчет

In [ ]:
def manual_line_count(filename):
    count_30_40 = 0
    examples = []

    with open(
        filename,
        "r",
        encoding="utf-8"
    ) as file:
        for line_number, line in enumerate(
            file,
            start=1
        ):
            clean_line = line.rstrip("\n")
            length = len(clean_line)

            if 30 <= length <= 40:
                count_30_40 += 1

                if len(examples) < 3:
                    examples.append({
                        "number": line_number,
                        "length": length,
                        "text": clean_line,
                    })

    return count_30_40, examples


manual_count, examples = manual_line_count(
    file_name
)

print(
    "Строк длиной 30-40 символов:",
    manual_count
)

for item in examples:
    print(
        f"Строка {item['number']} "
        f"({item['length']} символов): "
        f"{item['text']}"
    )

## 5. Статистика файла

In [ ]:
with open(
    file_name,
    "r",
    encoding="utf-8"
) as file:
    lines = [
        line.rstrip("\n")
        for line in file
    ]

total_lines = len(lines)
total_chars = sum(
    len(line)
    for line in lines
)

average_length = (
    total_chars / total_lines
    if total_lines
    else 0
)

percentage = (
    manual_count / total_lines * 100
    if total_lines
    else 0
)

print("Всего строк:", total_lines)
print("Всего символов:", total_chars)
print(
    f"Средняя длина строки: "
    f"{average_length:.2f}"
)
print(
    f"Доля строк 30-40 символов: "
    f"{percentage:.2f}%"
)

In [ ]:
length_ranges = {
    "0-9": 0,
    "10-19": 0,
    "20-29": 0,
    "30-40": 0,
    "41-50": 0,
    "51+": 0,
}

for line in lines:
    length = len(line)

    if length <= 9:
        length_ranges["0-9"] += 1
    elif length <= 19:
        length_ranges["10-19"] += 1
    elif length <= 29:
        length_ranges["20-29"] += 1
    elif length <= 40:
        length_ranges["30-40"] += 1
    elif length <= 50:
        length_ranges["41-50"] += 1
    else:
        length_ranges["51+"] += 1


for name, count in length_ranges.items():
    print(
        f"{name}: {count}"
    )

## 6. Расширенный MapReduce-анализ

In [ ]:
extended_script = r"""
from mrjob.job import MRJob


class ExtendedLineAnalysis(MRJob):
    def mapper(self, _, line):
        clean_line = line.rstrip("\\n")
        length = len(clean_line)

        if 30 <= length <= 40:
            yield "30-40", 1

        if length < 10:
            yield "range_0_9", 1
        elif length < 20:
            yield "range_10_19", 1
        elif length < 30:
            yield "range_20_29", 1
        elif length <= 40:
            yield "range_30_40", 1
        elif length <= 50:
            yield "range_41_50", 1
        else:
            yield "range_51_plus", 1

        yield "total_chars", length
        yield "total_lines", 1

    def reducer(self, key, values):
        yield key, sum(values)


if __name__ == "__main__":
    ExtendedLineAnalysis.run()
"""

extended_script_name = "extended_analysis.py"

Path(extended_script_name).write_text(
    extended_script,
    encoding="utf-8"
)

print("Создан файл:", extended_script_name)

In [ ]:
extended_result = subprocess.run(
    [
        sys.executable,
        extended_script_name,
        file_name,
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
)

print(extended_result.stdout)

if extended_result.stderr.strip():
    print(extended_result.stderr)

## 7. Сводка

In [ ]:
print("Файл:", file_name)
print(
    "Размер:",
    os.path.getsize(file_name),
    "байт"
)
print("Всего строк:", total_lines)
print(
    "Строк длиной 30-40 символов:",
    manual_count
)
print(
    f"Доля от общего числа: "
    f"{percentage:.2f}%"
)

## Итог

In [ ]:
created_files = [
    file_name,
    script_name,
    extended_script_name,
]

print("Файлы проекта:")

for file in created_files:
    path = Path(file)

    if path.exists():
        print(
            f"{path.name}: "
            f"{path.stat().st_size} байт"
        )